# Sunspot Analysis & Prediction

This notebook uses the `src` modules to load data, engineer features, train the Hybrid Ridge + LightGBM + EVT model, and export artefacts for the Gradio app.

For exploratory data analysis, see `00-EDA.ipynb`.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# --- Environment Setup ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/My Drive/Sunspots'
    if project_path not in sys.path:
        sys.path.append(project_path)
    os.chdir(project_path)
    print(f"Running in Colab. Directory: {os.getcwd()}")
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        sys.path.append(os.path.abspath('..'))
        os.chdir('..')
    print(f"Running locally. Directory: {os.getcwd()}")

from src.utils import load_config, plot_sunspots_series, plot_predictions
from src.data import load_data
from src.features import create_features, prepare_target
from src.train import train_evaluate, run_future_forecast

## 1. Load Data

Data is downloaded and explored in `00-EDA.ipynb`. Run that notebook first.  
`load_data()` checks for a local cache (`data/raw/sunspots.csv`) and skips re-downloading if it already exists.


In [ ]:
config = load_config('config.yaml')
df = load_data(config['data']['url'], save_path=config['data']['save_path'])
df.head()

## 2. Define Evaluation Period
Choose the range of data to use for features and evaluation.

In [ ]:
start_period = '1965-01-01'
end_period = df.index.max().strftime('%Y-%m-%d')

df_filtered = df.loc[start_period:end_period].copy()
print(f"Data points in period: {len(df_filtered)}")
plot_sunspots_series(df_filtered, title=f"Sunspots from {start_period} to {end_period}")

## 3. Training & Evaluation
Using Walk-Forward Validation.

In [ ]:
df_feat = create_features(df_filtered, config['features']['lags'], config['features']['rolling_windows'])
df_feat = prepare_target(df_feat, shift=config['features']['target_shift'])

X = df_feat.drop(columns=['target', 'SUNSPOTS', 'LOG_SUNSPOTS'])
y = df_feat['target']

results = train_evaluate(X, y, config)

print(f"Evaluation results for period {start_period} to {end_period}:")
print(f"- Hybrid RMSE: {results['hybrid_rmse']:.4f}")
print(f"- Hybrid MAE:  {results['hybrid_mae']:.4f}")

## 4. Baseline Comparison

To give the metrics meaning, we compare against established baselines using the **same walk-forward framework** (no data leakage):

| Baseline | Granularity | Description |
|---|---|---|
| **Naive (lag-1)** | Daily | Predict tomorrow = today |
| **Rolling mean (30d)** | Daily | Predict tomorrow = 30-day trailing average |
| **McNish-Lincoln (approx.)** | Monthly | 13-month trailing smooth — the method SIDC uses for official forecasts |
| **ARIMA(5,1,0)** | Monthly | Classical time series model, standard in solar forecasting literature |

*Daily baselines are evaluated on the same 5-day horizon as our hybrid model. Monthly baselines are evaluated month-ahead on resampled data — a standard benchmark in the literature.*


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA
from src.train import expanding_walk_forward_splits
import warnings
warnings.filterwarnings('ignore')

# ── Daily baselines (same walk-forward as hybrid) ────────────────────────────
naive_rmse, naive_mae = [], []
roll30_rmse, roll30_mae = [], []

for Xtr, ytr, Xval, yval in expanding_walk_forward_splits(
    X, y,
    config['evaluation']['initial_train_size'],
    config['evaluation']['val_size'],
    config['evaluation']['step_size']
):
    if len(Xtr) < 365:
        continue
    yval_linear = np.expm1(yval)
    naive_pred = np.full(len(yval), np.expm1(ytr.iloc[-1]))
    roll30_pred = np.full(len(yval), np.expm1(ytr.iloc[-30:].mean()))

    naive_rmse.append(np.sqrt(mean_squared_error(yval_linear, naive_pred)))
    naive_mae.append(mean_absolute_error(yval_linear, naive_pred))
    roll30_rmse.append(np.sqrt(mean_squared_error(yval_linear, roll30_pred)))
    roll30_mae.append(mean_absolute_error(yval_linear, roll30_pred))

# ── Monthly baselines: McNish-Lincoln & ARIMA ────────────────────────────────
# Resample to monthly means (how SIDC operates)
monthly = df_filtered['SUNSPOTS'].resample('MS').mean().dropna()

INIT_MONTHS = 132   # 11-year warm-up
ARIMA_WINDOW = 120  # sliding 10-year window keeps ARIMA fast

mcnish_rmse, mcnish_mae = [], []
arima_rmse,  arima_mae  = [], []

for i in range(INIT_MONTHS, len(monthly) - 1):
    train = monthly.iloc[:i]
    actual = monthly.iloc[i]

    # McNish-Lincoln: 13-month trailing smooth (SIDC approximation)
    mcnish_pred = train.iloc[-13:].mean()
    mcnish_rmse.append((actual - mcnish_pred) ** 2)
    mcnish_mae.append(abs(actual - mcnish_pred))

    # ARIMA(5,1,0) on sliding window
    try:
        fit = ARIMA(train.iloc[-ARIMA_WINDOW:], order=(5, 1, 0)).fit()
        arima_pred = max(fit.forecast(steps=1).iloc[0], 0)
        arima_rmse.append((actual - arima_pred) ** 2)
        arima_mae.append(abs(actual - arima_pred))
    except Exception:
        pass

# ── Results table ─────────────────────────────────────────────────────────────
hyb_r, hyb_m = results['hybrid_rmse'], results['hybrid_mae']

print(f"{'Model':<28} {'Granularity':<12} {'RMSE':>8} {'MAE':>8}")
print("─" * 60)
print(f"{'Naive (lag-1)':<28} {'daily':<12} {np.mean(naive_rmse):>8.2f} {np.mean(naive_mae):>8.2f}")
print(f"{'Rolling mean (30d)':<28} {'daily':<12} {np.mean(roll30_rmse):>8.2f} {np.mean(roll30_mae):>8.2f}")
print(f"{'McNish-Lincoln (13m smooth)':<28} {'monthly':<12} {np.sqrt(np.mean(mcnish_rmse)):>8.2f} {np.mean(mcnish_mae):>8.2f}")
print(f"{'ARIMA(5,1,0)':<28} {'monthly':<12} {np.sqrt(np.mean(arima_rmse)):>8.2f} {np.mean(arima_mae):>8.2f}")
print(f"{'Hybrid (ours)':<28} {'daily':<12} {hyb_r:>8.2f} {hyb_m:>8.2f}")
print()
naive_r, naive_m = np.mean(naive_rmse), np.mean(naive_mae)
print(f"Improvement over naive — RMSE: -{(naive_r-hyb_r)/naive_r*100:.1f}%  |  MAE: -{(naive_m-hyb_m)/naive_m*100:.1f}%")


## 5. 30-Day Horizon Experiment

Same model and same walk-forward framework, but now targeting **30 days ahead** instead of 5.

At longer horizons the short-lag autocorrelation that makes naive lag-1 so competitive decays significantly — this is where the ML model should show a clearer advantage. Monthly baselines (McNish-Lincoln, ARIMA) are directly comparable here since they also operate at ~monthly granularity.


In [ ]:
import copy

# ── Config for 30-day horizon ─────────────────────────────────────────────────
config_30d = copy.deepcopy(config)
config_30d['features']['target_shift'] = -30
config_30d['evaluation']['val_size']   = 30

# Re-prepare features with 30-day target
df_feat_30d = create_features(df_filtered, config_30d['features']['lags'], config_30d['features']['rolling_windows'])
df_feat_30d = prepare_target(df_feat_30d, shift=config_30d['features']['target_shift'])

X_30d = df_feat_30d.drop(columns=['target', 'SUNSPOTS', 'LOG_SUNSPOTS'])
y_30d = df_feat_30d['target']

# ── Train hybrid at 30-day horizon ───────────────────────────────────────────
print("Training hybrid model at 30-day horizon...")
results_30d = train_evaluate(X_30d, y_30d, config_30d)
print(f"Hybrid 30d — RMSE: {results_30d['hybrid_rmse']:.4f}  MAE: {results_30d['hybrid_mae']:.4f}")

# ── Daily baselines at 30-day horizon ────────────────────────────────────────
naive30_rmse, naive30_mae = [], []
roll30_30d_rmse, roll30_30d_mae = [], []

for Xtr, ytr, Xval, yval in expanding_walk_forward_splits(
    X_30d, y_30d,
    config_30d['evaluation']['initial_train_size'],
    config_30d['evaluation']['val_size'],
    config_30d['evaluation']['step_size']
):
    if len(Xtr) < 365:
        continue
    yval_linear = np.expm1(yval)
    naive_pred  = np.full(len(yval), np.expm1(ytr.iloc[-1]))
    roll_pred   = np.full(len(yval), np.expm1(ytr.iloc[-30:].mean()))

    naive30_rmse.append(np.sqrt(mean_squared_error(yval_linear, naive_pred)))
    naive30_mae.append(mean_absolute_error(yval_linear, naive_pred))
    roll30_30d_rmse.append(np.sqrt(mean_squared_error(yval_linear, roll_pred)))
    roll30_30d_mae.append(mean_absolute_error(yval_linear, roll_pred))

# ── Results table ─────────────────────────────────────────────────────────────
hyb30_r, hyb30_m = results_30d['hybrid_rmse'], results_30d['hybrid_mae']
n30_r,   n30_m   = np.mean(naive30_rmse),      np.mean(naive30_mae)
r30_r,   r30_m   = np.mean(roll30_30d_rmse),   np.mean(roll30_30d_mae)
mcn_r,   mcn_m   = np.sqrt(np.mean(mcnish_rmse)), np.mean(mcnish_mae)
arm_r,   arm_m   = np.sqrt(np.mean(arima_rmse)),   np.mean(arima_mae)

print()
print(f"{'Model':<28} {'Granularity':<12} {'RMSE':>8} {'MAE':>8}")
print("─" * 60)
print(f"{'Naive (lag-1)':<28} {'daily':<12} {n30_r:>8.2f} {n30_m:>8.2f}")
print(f"{'Rolling mean (30d)':<28} {'daily':<12} {r30_r:>8.2f} {r30_m:>8.2f}")
print(f"{'McNish-Lincoln (13m smooth)':<28} {'monthly':<12} {mcn_r:>8.2f} {mcn_m:>8.2f}")
print(f"{'ARIMA(5,1,0)':<28} {'monthly':<12} {arm_r:>8.2f} {arm_m:>8.2f}")
print(f"{'Hybrid (ours, 30d)':<28} {'daily':<12} {hyb30_r:>8.2f} {hyb30_m:>8.2f}")
print()

def delta(baseline, model):
    pct = (baseline - model) / baseline * 100
    return f"{pct:+.1f}%"

print(f"vs Naive (lag-1)   — RMSE: {delta(n30_r, hyb30_r)}  MAE: {delta(n30_m, hyb30_m)}")
print(f"vs Roll mean (30d) — RMSE: {delta(r30_r, hyb30_r)}  MAE: {delta(r30_m, hyb30_m)}")
print(f"vs ARIMA(5,1,0)    — RMSE: {delta(arm_r, hyb30_r)}  MAE: {delta(arm_m, hyb30_m)}")
print()
print("Note: positive = hybrid wins, negative = baseline wins.")


## 5. Visualizing Predictions vs Real
Walk-forward predictions vs actual values.

In [ ]:
plt.figure(figsize=(15, 5))
plt.plot(results['actuals'], label='Actual', alpha=0.6, color='blue')
plt.plot(results['predictions'], label='Forecast', alpha=0.8, color='red', linestyle='--')
plt.title("Walk-Forward Forecast vs Real Values")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Future Forecasting
Make a prediction for multiple days ahead.

In [ ]:
steps_ahead = 5

future_predictions = run_future_forecast(df_filtered, steps=steps_ahead, config=config)

print("date           sunspots")
for date, val in future_predictions.items():
    print(f"{date.strftime('%d/%m/%Y')}   {val:.2f}")

## 7. Export Models
Saves the config and filtered data so the Gradio app can reconstruct features and refit models on the fly.

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(df_filtered, 'models/sunspots_data.joblib')
joblib.dump(config, 'models/config.joblib')
joblib.dump(results, 'models/validation_results.joblib')
print("Model data, config, and validation results exported to /models")